In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import cobra
from cobra.core import configuration
from PolyRound.api import PolyRoundApi
from PolyRound.settings import PolyRoundSettings
import hopsy

In [88]:
# configuration.Configuration.solver = 'glpk'

In [7]:
model = cobra.io.read_sbml_model('./../autopacmen_output/Mitocore_aligned_to_Human1/MitoCore_aligned_to_Human1_calibrated_MA.xml')

In [8]:
model.optimize()

,fluxes,reduced_costs
EX_2hb_e,0.000000e+00,0.0
EX_ac_e,-1.151314e-01,0.0
EX_acac_e,0.000000e+00,0.0
EX_akg_e,0.000000e+00,0.0
EX_ala_B_e,0.000000e+00,0.0
...,...,...
ENZYME_DELIVERY_ENSG00000058063,0.000000e+00,0.0
ENZYME_DELIVERY_ENSG00000124406,0.000000e+00,0.0
ENZYME_DELIVERY_ENSG00000085231,4.790755e-08,0.0
ENZYME_DELIVERY_ENSG00000102743,0.000000e+00,0.0


# 1) Sanity check: Model has a solution with lower bound for objective function

In [11]:
for reaction in model.reactions:
    if reaction.id.startswith('ENZYME_DELIVERY_'):
        if len(list(reaction.metabolites.keys())[0].reactions) == 1:
            model.remove_metabolites(list(reaction.metabolites.keys()))
            model.remove_reactions(reaction)
            print(f'removed reaction {reaction.id}')

model.reactions.OF_ATP_MitoCore.lower_bound = 1.707042254 * 0.95
model.optimize()

removed reaction ENZYME_DELIVERY_ENSG00000169692
removed reaction ENZYME_DELIVERY_ENSG00000141526
removed reaction ENZYME_DELIVERY_ENSG00000178537
removed reaction ENZYME_DELIVERY_ENSG00000115840
removed reaction ENZYME_DELIVERY_ENSG00000143158
removed reaction ENZYME_DELIVERY_ENSG00000198712
removed reaction ENZYME_DELIVERY_ENSG00000168003
removed reaction ENZYME_DELIVERY_ENSG00000164919
removed reaction ENZYME_DELIVERY_ENSG00000135940
removed reaction ENZYME_DELIVERY_ENSG00000111775
removed reaction ENZYME_DELIVERY_ENSG00000131143
removed reaction ENZYME_DELIVERY_ENSG00000112695
removed reaction ENZYME_DELIVERY_ENSG00000126267
removed reaction ENZYME_DELIVERY_ENSG00000127184
removed reaction ENZYME_DELIVERY_ENSG00000178741
removed reaction ENZYME_DELIVERY_ENSG00000066926
removed reaction ENZYME_DELIVERY_ENSG00000157184
removed reaction ENZYME_DELIVERY_ENSG00000110090
removed reaction ENZYME_DELIVERY_ENSG00000100075
removed reaction ENZYME_DELIVERY_ENSG00000075415
removed reaction ENZ

c:\Users\emanuel.lange\.conda\envs\hopsy-sampling\lib\site-packages\cobra\core\model.py:779: UserWarning: need to pass in a list
  warn("need to pass in a list")


,fluxes,reduced_costs
EX_2hb_e,0.000000e+00,-0.0
EX_ac_e,-1.151314e-01,-0.0
EX_acac_e,0.000000e+00,-0.0
EX_akg_e,0.000000e+00,-0.0
EX_ala_B_e,0.000000e+00,-0.0
...,...,...
ENZYME_DELIVERY_ENSG00000151498,0.000000e+00,0.0
ENZYME_DELIVERY_ENSG00000197142,0.000000e+00,0.0
ENZYME_DELIVERY_ENSG00000124406,0.000000e+00,0.0
ENZYME_DELIVERY_ENSG00000085231,4.790755e-08,0.0


In [12]:
cobra.io.write_sbml_model(model, './../flux_sampling/chrr_reounding_test_h1_ma.xml')

In [15]:
polytope_orphan = PolyRoundApi.sbml_to_polytope('./../flux_sampling/chrr_reounding_test_h1_ma.xml')

# reinitialize hopsies settings for polyround to use glpk solver
settings = PolyRoundSettings(verbose=True)
hopsy.LP.settings = settings
hopsy.LP.settings.__dict__

problem = hopsy.Problem(polytope_orphan.A, polytope_orphan.b) # set the polytype, i.e., the inequality constraints from the constraint based model
problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope_orphan.S, b_eq=polytope_orphan.h)
start = time.perf_counter()
problem = hopsy.round(problem)
# print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

Using the hp flags: {'FeasibilityTol': 1e-09, 'OptimalityTol': 1e-08}

 Investigating constraint number: 0


 Investigating constraint number: 50


 Investigating constraint number: 100


 Investigating constraint number: 150


 Investigating constraint number: 200


 Investigating constraint number: 250


 Investigating constraint number: 300


 Investigating constraint number: 350


 Investigating constraint number: 400


 Investigating constraint number: 450


 Investigating constraint number: 500


 Investigating constraint number: 550


 Investigating constraint number: 600


 Investigating constraint number: 650


 Investigating constraint number: 700


 Investigating constraint number: 750


 Investigating constraint number: 800


 Investigating constraint number: 850


 Investigating constraint number: 900


 Investigating constraint number: 950


 Investigating constraint number: 1000


 Investigating constraint number: 1050


 Investigating constraint number: 1100


 Investig

# 2) Reproduction of the Error 

In [93]:
# reinitialize hopsies settings for polyround to use glpk solver
# settings = PolyRoundSettings(backend='glpk')
# hopsy.LP.settings = settings
# hopsy.LP.settings.__dict__

In [ ]:
polytope_og = PolyRoundApi.sbml_to_polytope('./../flux_sampling/Mitocore_Original_plt_sampling_model.xml')
# problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
# problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
# start = time.perf_counter()
# problem = hopsy.round(problem)
# print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

In [95]:
polytope_mm = PolyRoundApi.sbml_to_polytope('./../flux_sampling/Mitocore_MitoMammal_sampling_model.xml')
# problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
# problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
# start = time.perf_counter()
# problem = hopsy.round(problem)
# print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

In [111]:
polytope_h1 = PolyRoundApi.sbml_to_polytope('./../flux_sampling/Mitocore_aligned_to_Human1_sampling_model.xml')
# problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
# problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
# start = time.perf_counter()
# problem = hopsy.round(problem)
# print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

In [97]:
from scipy.sparse.linalg import svds
import numpy as np

In [ ]:
def get_constraints_ratio(b):
    lower_bound_end_index = int(len(b)/2)
    delta = np.subtract(b.iloc[0:lower_bound_end_index].values,b.iloc[lower_bound_end_index:].values)
    delta = [value for value in delta if value > 0]
    const_ratio = max(delta) / min(delta)
    print('constraint ratio: ', const_ratio)
    
    return const_ratio

In [115]:
def get_condition_number(S):
    
    # Compute all singular values (s)
    # compute_uv=False makes it faster as we don't need the vectors
    s = np.linalg.svd(S, compute_uv=False)

    # Filter out values that are effectively zero (within machine precision)
    # This prevents an 'Infinite' condition number due to noise
    tol = s.max() * max(S.shape) * np.finfo(s.dtype).eps
    s_nonzero = s[s > tol]

    if len(s_nonzero) < min(S.shape):
        print(f"Warning: Matrix is rank deficient! Rank = {len(s_nonzero)}")

    cond_S = s_nonzero.max() / s_nonzero.min()
    print(f"Condition Number: {cond_S:.2f}")
    
    return cond_S

In [116]:
cn_og = get_condition_number(polytope_og.S.to_numpy())
cn_mm = get_condition_number(polytope_mm.S.to_numpy())
cn_h1 = get_condition_number(polytope_h1.S.to_numpy())  

Condition Number: 150.53
Condition Number: 90134.68
Condition Number: 100708.64


In [117]:
print(cn_og/cn_og)
print(cn_mm/cn_og)
print(cn_h1/cn_og)

1.0
598.7970255108971
669.0436736387574


In [ ]:
cr_og = get_constraints_ratio(polytope_og.b)
cr_mm = get_constraints_ratio(polytope_mm.b)
cr_h1 = get_constraints_ratio(polytope_h1.b)

print(cr_og/cr_og)
print(cr_mm/cr_og)
print(cr_h1/cr_og)

contraint ratio:  4173423.708922083
contraint ratio:  5139184.598967615
contraint ratio:  16693694.835688332
1.0
1.231407342604801
4.0


: 

In [4]:
def show_min_max_constraints(model):
    max_constraint = 0
    min_constraint = 1000
    max_stoichiometry = 1
    reaction_min = ''
    reaction_max = ''

    for reaction in model.reactions:
        delta = reaction.upper_bound - reaction.lower_bound
        if delta == 0:
            continue
        if delta > max_constraint:
            max_constraint = delta
            reaction_max = reaction.id
        if delta < min_constraint:
            min_constraint = delta
            reaction_min = reaction.id
        
        for stoichiometry in reaction.metabolites.values():
            if abs(stoichiometry) > max_stoichiometry:
                max_stoichiometry = abs(stoichiometry)

    print(f'{reaction_min}: {min_constraint}')
    print(f'{reaction_max}: {max_constraint}')
    
    return max_constraint, max_stoichiometry

In [110]:
og_model = cobra.io.read_sbml_model('./../flux_sampling/Mitocore_Original_plt_sampling_model.xml')
mm_model = cobra.io.read_sbml_model('./../flux_sampling/Mitocore_MitoMammal_sampling_model.xml')
h1_model = cobra.io.read_sbml_model('./../flux_sampling/Mitocore_aligned_to_Human1_sampling_model.xml')

show_min_max_constraints(og_model)
show_min_max_constraints(mm_model)
show_min_max_constraints(h1_model)

ILEtec: 0.00024
EX_2hb_e: 2000.0
ENZYME_DELIVERY_ENSG00000151366: 0.000194898951546226
EX_2hb_e: 2000.0
CYStec_TG_forward: 6e-05
EX_2hb_e: 2000.0


# 2) Algorithm to scale extreme enzyme constraints and stoichiometries

In [14]:
def scale_metabolite_stoichiomtry(model, metabolite, scaling_factor):
    for reaction in model.reactions:
        metabolite_dict = reaction.metabolites
        
        if not metabolite in metabolite_dict:
            continue
        
        stoichiometric_coefficient = reaction.metabolites[metabolite]
        metabolite_dict[metabolite] = stoichiometric_coefficient * scaling_factor
    
    reaction.add_metabolites(metabolite_dict, combine = False)
        

In [38]:
import math

def scale_constraints_and_stoichiometries(model):
    
    # get largest range of constraints in model    
    max_constraint_range, max_stoichiometry = show_min_max_constraints(model)
    
    orphan_deliveries = []
    
    with model:
        for reaction in model.reactions:
            if not (reaction.id.startswith('ENZYME_DELIVERY_') or reaction.id == 'ER_pool_TG_'):
                continue
            
            # get upper limit for scaling based on enzyme constraint
            max_constraint_scaling_factor = math.floor(max_constraint_range / reaction.upper_bound)
            
            # get pseudometabolite
            metabolite = list(reaction.metabolites.keys())[0]
            
            # get largest stoichiometry in reactions connected to pseudometabolite
            stoichiometries = []
            
            for met_reaction in metabolite.reactions:
                if met_reaction.id == reaction.id:
                    continue 
                
                stoichiometries.append(abs(met_reaction.metabolites[metabolite]))
            
            if len(stoichiometries) == 0:
                orphan_deliveries.append(reaction.id)
                model.remove_reactions(reaction)
                model.remove_metabolites(list(reaction.metabolites.keys()))
                print(f'Reaction {reaction.id} is an orphan delivery!')
                continue
            
            max_met_stoichiometry = max(stoichiometries)
            
            if max_met_stoichiometry == max_stoichiometry:
                continue
            
            max_stoich_scaling = math.floor(max_stoichiometry / max_met_stoichiometry)
            
            if reaction.id == 'ENZYME_DELIVERY_ENSG00000151366':
                print('hello')
            
            scaling_factor = max_constraint_scaling_factor if max_constraint_scaling_factor < max_stoich_scaling else max_stoich_scaling
            
            reaction.upper_bound = reaction.upper_bound * scaling_factor
            
            # scale stoichiometries in connected reactions
            scale_metabolite_stoichiomtry(model, metabolite, scaling_factor)

            # sanity check
        print(model.optimize())
        
        show_min_max_constraints(model)

In [39]:
# explore production near optimum: set lower bound of ATP production to mean measured value = 1.707 --> explore ATP production near mean
atp_production_lower_bound = 1.707042254 * 0.95

with model:
    model.reactions.OF_ATP_MitoCore.lower_bound = atp_production_lower_bound
    
    print(model.optimize())
    show_min_max_constraints(model)

    scale_constraints_and_stoichiometries(model)
    


<Solution 2.217 at 0x20828cda170>
ENZYME_DELIVERY_ENSG00000151366: 1.94898951546226e-07
EX_2hb_e: 2000.0
ENZYME_DELIVERY_ENSG00000151366: 1.94898951546226e-07
EX_2hb_e: 2000.0
hello
Reaction ENZYME_DELIVERY_ENSG00000102743 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000112697 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000143653 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000108528 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000157184 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000126267 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000135940 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000143158 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000141526 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000198712 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000111775 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000127184 is an orphan delivery!
Reaction ENZYME_DELIVERY_ENSG00000110090 is an orp